# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !uv venv .venv --seed

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Mount Google Drive
Connect to Google Drive so you can load the dataset and save your results.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Run the cell below every time to activate the installed environment.

In [3]:
!pip uninstall -y torch torchvision torchaudio transformers protobuf tensorflow tensorflow-cpu
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q vllm==0.8.5 transformers==4.51.3 accelerate bitsandbytes tqdm sympy antlr4-python3-runtime==4.11.1
!pip install -q protobuf==4.25.3

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Found existing installation: transformers 4.51.3
Uninstalling transformers-4.51.3:
  Successfully uninstalled transformers-4.51.3
Found existing installation: protobuf 4.25.3
Uninstalling protobuf-4.25.3:
  Successfully uninstalled protobuf-4.25.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xgrammar 0.1.18 requires transformers>=4.38.0, which is not installed.
compressed-tensors 0.9.3 requires transformers, which is not installed.
vllm 0.8.5 requires protob

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [4]:
import torch
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("CUDA is not available. Please connect to a GPU runtime (Runtime > Change runtime type).")

NVIDIA A100-SXM4-80GB


In [5]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES (Changed from 1 to 0 for Colab)
DATA_PATH   = "/content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition/data/public.jsonl"
OUTPUT_PATH = "/content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition/results/frq_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

INFO 05-31 22:50:22 [__init__.py:239] Automatically detected platform cuda.


In [6]:
# DATA_PATH   = "/content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition/data/private.jsonl"
# data = [json.loads(line) for line in open(DATA_PATH)]
# len(data)

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [7]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 1126 questions  (375 MCQ, 751 free-form)

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
}

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [8]:
PROMPT_VARIANTS = {
    "multiple_answers": {
        "math": (
            "Solve the math problem. Show only the necessary reasoning. "
            "Final answer rules:"
            "- Include EVERY requested answer in the final answer."
            "- If the problem has multiple parts or multiple blanks, put all answers in ONE final \\boxed{...}, separated by commas."
            "- Use exact form."
            "- No decimal approximations."
            "- Keep expressions symbolic."
            "- Use \\frac{}{}, powers, \\sqrt{}, \\pi, \\ln{}, \\arctan{} when appropriate."
            "- Only use decimals for numbers that are already decimals in the problem."
            "- End with the final answer in \\boxed{...}."
        ),
        "mcq": (
            "Solve the multiple-choice math problem. "
            "Output ONLY the correct choice letter inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },
    "qwen_safeguard": {
        "math": (
            "You are an expert mathematician. Solve the problem carefully but concisely.\n\n"
            "Important final-answer rules:\n"
            "- Include EVERY requested answer in the final answer.\n"
            "- If the problem has multiple parts, multiple blanks, or multiple requested values, "
            "put all answers in ONE final \\boxed{...}, separated by commas.\n"
            "- Match the requested answer type for each blank/subpart.\n"
            "- If a blank asks for an expression or formula, keep variables symbolic; do not substitute numbers.\n"
            "- If a blank asks for a numerical value, compute it.\n"
            "- Use exact form when appropriate.\n"
            "- Do not round unless the problem specifies a precision. If a decimal answer is required and no precision is specified, give at least 12 significant digits.\n"
            "- Keep fractions, powers, roots, logarithms, inverse trig, and pi symbolic when exact form is appropriate.\n"
            "- For tangent equations, prefer arctan(value), pi instead of decimal approximations unless decimals are explicitly required.\n"
            "- Use parser-friendly notation in the final answer: use * for multiplication and ^ for powers.\n"
            "- Write e^(16*x), not e^{16x}; write 6*e^(16*x), not 6e^{16x}.\n"
            "- For yes/no answers, write YES or NO.\n"
            "- For letter-set answers, write letters together with no spaces, e.g. BCEG.\n"
            "- End with exactly one final answer in \\boxed{...}."
        ),
        "mcq": (
            "You are an expert mathematician. Solve the multiple-choice problem carefully but concisely.\n"
            "Choose the single best answer choice.\n"
            "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
        ),
    },
    # "frq_blanks": {
    #     "math": (
    #         "You are an expert mathematician. Solve carefully but concisely.\n"
    #         "If the problem contains [ANS] blanks, treat each [ANS] as a blank to fill and return the blank values in order.\n"
    #         "If the problem does not contain [ANS], solve for the requested final answer.\n\n"
    #         "Final answer rules:\n"
    #         "- Include every requested answer in one final \\boxed{...}.\n"
    #         "- If there are multiple [ANS] blanks or subparts, put the answers in the same order as they appear.\n"
    #         "- Match the requested format near each blank/subpart.\n"
    #         "- If the problem says to round, use the requested rounding.\n"
    #         "- If it asks for significant digits, decimals are acceptable.\n"
    #         "- If a blank asks for an expression or formula, keep variables symbolic.\n"
    #         "- If a blank asks for a numerical value, compute it.\n"
    #         "- Use parser-friendly notation in the final answer: use * for multiplication and ^ for powers.\n"
    #         "- For yes/no answers, write YES or NO.\n"
    #         "- For letter-choice blanks, write only the choice letter.\n"
    #         "- Do not include units in the boxed answer.\n"
    #         "- End with exactly one \\boxed{...}."
    #     ),
    #     "mcq": (
    #         "You are an expert mathematician. Solve the multiple-choice problem carefully but concisely.\n"
    #         "Choose the single best answer choice.\n"
    #         "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
    #     ),
    # },
    # "linus": {
    #     "math": (
    #         "You are an expert mathematician and The Fields Medal Winner competing in a high-stakes automated grading competition.\n"
    #         "You must solve the problem and format your answer perfectly to pass a strict string-matching judger.\n\n"
    #         "1. **Reasoning Phase (<think>):** Enclose all scratchpad work inside <think>...</think> tags.\n"
    #         "2. **Algorithmic Arithmetic:** To prevent calculation errors, you must decompose complex calculations. Inside your <think> block:\n"
    #         "    a. Isolate every root, logarithm, or fraction.\n"
    #         "    b. Retrieve the highly precise decimal value (to 12 digits after decimals) for each isolated component from your training memory BEFORE multiplying or dividing them. ex. 12.500000000000\n"
    #         "    c. Perform operations one single step at a time.\n"
    #         "3. **High-Precision Evaluation:** Your absolute final answer MUST be evaluated into a precise floating-point decimal to match the judger's expectations.\n"
    #         "4. **FORMATTING FATALITIES (CRITICAL):** The judger uses strict string matching. You will score 0 if you fail these rules:\n"
    #         "    * NO trailing zeros. (If the answer is 2.500, you MUST output 2.5).\n"
    #         "    * NO units, words, or variables inside the box (e.g., NO 'F', 'hours', 't=').\n"
    #         "    * ONLY numbers and commas inside the \\boxed{}.\n"
    #         "    * BAD: \\boxed{143.224229233795, t = 2.32624773420025}\n"
    #         "    * GOOD: \\boxed{143.224229233795, 2.32624773420025}\n"
    #         "5. **Overrides:** If the specific question explicitly asks for a fraction, radical, or exact terms of pi, follow that requirement instead of the decimal rule.\n"
    #         "6. **Strict Placement:** The \\boxed{} tag must be the absolute final element of your response, outside of the <think> tags."
    #         "7. Make sure to keep ur response concise and after you get ur final answer, make sure to re-evaluate using your final answer to see if it making sense.\n"
    #         "8. Remeber all the numbers and steps you used, so you don't make up a number randomly."
    #         "9. Make sure you end the reasoning with your final answer inside \\boxed{}, this is very important!"
    #         "10. Ensure you keep all reasoning and inference step reasonable and double-check the validity"
    #         "11. Proof read your reasoning and check for any nonsense, keep it concise"
    #     ),
    #     "mcq": (
    #         "You are an expert mathematician and The Fields Medal Winner."
    #         "Read the problem and the answer choices below, then select the single best answer. "
    #         "Maintain exact precision. Leave unsimplifiable fractions, radicals, or exponents as exact mathematical expressions rather than calculating decimals."
    #         "If instructed by question then you have to simplified the decimals, fractions etc, otherwise no"
    #         "Keep your response concise and reasonable"
    #         "Review ur reasoning to make sure your answer(s) is correct"
    #         "You have to make sure your final answer is \\boxed{}, e.g. \\boxed{C}."
    #         "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
    #     ),
    # }
}


In [9]:
EXAMPLE_FRQ_USER = (
    "Fill in the blanks.\n"
    "(a) If x=2.5, compute 7*x^2 + 1. Your answer is [ANS].\n"
    "(b) Solve 0.96584^t = 0.5 for t. Give an exact expression. Your answer is [ANS].\n"
    "(c) Determine all solutions for tan(theta)=4.76 in the form theta=[ANS]+[ANS]n.\n"
    "(d) Is 7*x^2 + 1 greater than 40 when x=2.5? Your answer is [ANS]."
)

EXAMPLE_FRQ_ASSISTANT = (
    "<think>\n"
    "There are five [ANS] blanks, so the final answer must contain five values in order.\n"
    "For (a), compute directly: 7*2.5^2 + 1 = 44.75.\n"
    "For (b), solve exactly: 0.96584^t=0.5 gives t=ln(0.5)/ln(0.96584). Do not decimalize because an exact expression is requested.\n"
    "For (c), tangent has period pi, so theta=arctan(4.76)+pi*n. The two blanks are arctan(4.76) and pi.\n"
    "For (d), 44.75>40, so the answer is YES.\n"
    "</think>\n"
    "\\boxed{44.75, ln(0.5)/ln(0.96584), arctan(4.76), pi, YES}"
)

EXAMPLE_MCQ_USER = (
    "What is 12 + 7?\n"
    "A. 17\n"
    "B. 18\n"
    "C. 19\n"
    "D. 20"
)

EXAMPLE_MCQ_ASSISTANT = (
    "<think>\n"
    "12+7=19, which corresponds to choice C.\n"
    "</think>\n"
    "\\boxed{C}"
)

In [10]:
def build_prompt(
    question: str,
    options: Optional[list],
    variant: str = "baseline"
    )-> tuple[str, str]:

    """Return (system_prompt, user_prompt) for a question."""

    prompts = PROMPT_VARIANTS[variant]

    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return prompts['mcq'], f"{question}\n\nOptions:\n{opts_text}"
    return prompts['math'], question

## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 05-31 22:50:47 [config.py:717] This model supports multiple tasks: {'reward', 'score', 'embed', 'generate', 'classify'}. Defaulting to 'generate'.
WARNING 05-31 22:50:47 [config.py:830] bitsandbytes quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 05-31 22:50:47 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=32768.
WARNING 05-31 22:50:50 [utils.py:2382] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/getting_started/troubleshooting.html#python-multiprocessing for more information. Reason: CUDA is initialized
INFO 05-31 22:52:24 [core_client.py:439] Core engine process 0 ready.
Model loaded.


## 5. Load Model with Transformers (alternative to vLLM for DataHub)

We load **Qwen3-4B-Thinking-2507** with **INT4 quantization** via BitsAndBytes.  

Key parameters:
- `load_in_4bit` — quantization strategy of INT4

In [12]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = 'left'

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# llm = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [13]:
# Build prompts for specific FRQ questions
test_ids = [2, 3, 5, 7, 8, 16]
N = 100
#frq_data = [data[i] for i in frq_ids]
test_data = data[:N]


In [14]:
# Build prompts for first N entries
prompts = {}
for variant in PROMPT_VARIANTS:
  prompts[variant] = []
  for item in test_data:
    system, user = build_prompt(item["question"], item.get("options"), variant)

    is_mcq = item.get("options") is not None

    if is_mcq:
      example_user = EXAMPLE_MCQ_USER
      example_assistant = EXAMPLE_MCQ_ASSISTANT
    else:
      example_user = EXAMPLE_FRQ_USER
      example_assistant = EXAMPLE_FRQ_ASSISTANT


    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": example_user},
        {"role": "assistant", "content": example_assistant},
        {"role": "user",   "content": user},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    prompts[variant].append(prompt_text)

In [15]:
responses = {}
finish_reasons = {}

for variant in PROMPT_VARIANTS:
    print(f"Generating responses for {len(prompts[variant])} questions for variant: {variant}...")

    outputs = llm.generate(prompts[variant], sampling_params=sampling_params)

    responses[variant] = [
        out.outputs[0].text.strip()
        for out in outputs
    ]

    finish_reasons[variant] = [
        out.outputs[0].finish_reason
        for out in outputs
    ]

Generating responses for 100 questions for variant: multiple_answers...


Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses for 100 questions for variant: qwen_safeguard...


Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses for 100 questions for variant: frq_blanks...


Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating responses for 100 questions for variant: linus...


Processed prompts:   0%|          | 0/100 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [16]:
# Preview first question for each variant
for variant in PROMPT_VARIANTS:
#for i in range(min(3, len(responses))):
    i = 0
    print(f"\n── Response {i} (id={data[i].get('id')}), (variant: {variant}) ──")
    print(responses[variant][i])
    #print(responses[variant][i][:400], "..." if len(responses[variant][i]) > 400 else "")


── Response 0 (id=0), (variant: multiple_answers) ──
Okay, let's see. The problem is to find the sum of the first 325 positive even whole numbers. Hmm, first I need to remember what the first positive even whole numbers are. Positive even whole numbers start from 2, 4, 6, 8, etc. So the first one is 2, the second is 4, the third is 6, and so on. 

I think there's a formula for the sum of the first n even numbers. Let me recall. The nth even number is 2n, right? Because the first even number is 2*1=2, the second is 2*2=4, so the kth even number is 2k. 

So the sum of the first n even numbers would be 2 + 4 + 6 + ... + 2n. That's an arithmetic series where the first term a1 is 2, the common difference d is 2, and the number of terms is n. 

The formula for the sum of an arithmetic series is S = n/2 * (a1 + an), where an is the last term. Here, an = 2n, so the sum would be n/2 * (2 + 2n) = n/2 * 2(n + 1) = n(n + 1). Wait, let me check that. For example, if n=1, sum is 2, and 1*(1+1)=2, w

In [17]:
from collections import Counter

for variant in PROMPT_VARIANTS:
    print(f"\nVariant: {variant}")
    print(Counter(finish_reasons[variant]))


Variant: multiple_answers
Counter({'stop': 94, 'length': 6})

Variant: qwen_safeguard
Counter({'stop': 96, 'length': 4})

Variant: frq_blanks
Counter({'stop': 96, 'length': 4})

Variant: linus
Counter({'stop': 96, 'length': 4})


In [18]:
# Compare answer lengths between the four variants

response_lengths = {} # To store lengths for each variant

for variant in PROMPT_VARIANTS:
    lengths = [len(response) for response in responses[variant]]
    response_lengths[variant] = lengths

print("Average Response Lengths per Variant:")
print("=" * 50)
for variant, lengths in response_lengths.items():
    if lengths:
        average_length = sum(lengths) / len(lengths)
        print(f"  {variant.ljust(15)}: {average_length:.2f} characters")
    else:
        print(f"  {variant.ljust(15)}: No responses generated.")
print("=" * 50)

Average Response Lengths per Variant:
  multiple_answers: 12268.34 characters
  qwen_safeguard : 12396.62 characters
  frq_blanks     : 12351.28 characters
  linus          : 15209.91 characters


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [19]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
import sys
sys.path.insert(0, "/content/drive/MyDrive/GitHub_Repos/151B_SP26_Competition")
from judger import Judger
judger = Judger(strict_extract=False)

results = {}
for variant in PROMPT_VARIANTS:
  results[variant] = []
  for item, response in tqdm(zip(test_data, responses[variant]), total=len(data), desc="Scoring"):
      is_mcq = bool(item.get("options"))
      gold   = item["answer"]

      if is_mcq:
          correct = score_mcq(response, str(gold))
      else:
          gold_list = gold if isinstance(gold, list) else [gold]
          try:
              correct = judger.auto_judge(
                  pred=response,
                  gold=gold_list,
                  options=[[]] * len(gold_list),
              )
          except Exception:
              correct = False

      results[variant].append({
          "id":       item.get("id"),
          "is_mcq":   is_mcq,
          "gold":     gold,
          "response": response,
          "correct":  correct,
      })

#print(f"Scoring complete. {len(results)} results.")
print("Scoring complete")

Scoring:   9%|▉         | 100/1126 [00:06<01:11, 14.41it/s]

Scoring complete


In [20]:
import pandas as pd
from judger import Judger

judger = Judger(strict_extract=False)

rows = []

for variant, result_list in results.items():
    for idx, r in enumerate(result_list):
        response = r["response"]
        extracted_raw = judger.extract_ans(response)

        if extracted_raw:
            extracted_split = judger.split_by_comma(extracted_raw)
            extracted_normalized = [
                judger.norm_ans_str(ans)
                for ans in extracted_split
            ]
        else:
            extracted_split = []
            extracted_normalized = []

        rows.append({
            "variant": variant,
            "idx": idx,
            "id": r.get("id"),
            "is_mcq": r.get("is_mcq"),
            "gold": r.get("gold"),
            "correct": r.get("correct"),
            "response": response,
            #"judger_extracted_raw": extracted_raw,
            #"judger_extracted_split": extracted_split,
            "judger_extracted_normalized": extracted_normalized,
            "response_chars": len(response) if isinstance(response, str) else None,
            'stop_reason': finish_reasons[variant][idx] if finish_reasons[variant][idx] else None
        })

debug_df = pd.DataFrame(rows)
debug_df.head(2)

,variant,idx,id,is_mcq,gold,correct,response,judger_extracted_normalized,response_chars,stop_reason
0,multiple_answers,0,0,False,[325*(1+325)],True,"Okay, let's see. The problem is to find the su...",[105950],3174,stop
1,multiple_answers,1,1,True,F,False,"Okay, let's try to figure out this integral. T...",[B],20420,stop


In [26]:
debug_df[(debug_df["correct"] == False) & (debug_df["variant"] == "qwen_safeguard")]

,variant,idx,id,is_mcq,gold,correct,response,judger_extracted_normalized,response_chars,stop_reason
101,qwen_safeguard,1,1,True,F,False,"Okay, let's try to figure out this integral: t...",[E],22812,stop
102,qwen_safeguard,2,2,False,"[143.224229233795, 2.32624773420025]",False,"Okay, let's tackle this problem step by step. ...","[143, 2.33]",13615,stop
105,qwen_safeguard,5,5,False,"[62.7777777777778, 335.927777777778, 604.67]",False,"Okay, let's tackle this problem step by step. ...","[62.78, 335.93, 604.67]",8637,stop
108,qwen_safeguard,8,8,False,[(1/2)^[(1999-1963)/31]],False,"Okay, let's try to figure out this problem. So...",[0.446],7834,stop
112,qwen_safeguard,12,12,False,"[380, 315, 13, 310]",False,"Okay, let's tackle these deer population probl...","[380, 315, 14, 310]",5034,stop
116,qwen_safeguard,16,16,False,"[atan(4.76), pi]",False,"Okay, let's tackle part (c) here. The question...","[1.363, 3.142]",16317,stop
120,qwen_safeguard,20,20,False,"[-10, 100, -9, 81, 1, 1, 3, 9, 7, 49, 8, 64, 3...",False,"Okay, let's tackle this problem step by step. ...",[7.797],9658,stop
122,qwen_safeguard,22,22,False,"[1.6, 1.76, 0.16*p/16, up, 1, 0.16]",False,"Okay, let's tackle this problem step by step. ...","[44.75, \ln(0.5)/\ln(0.96584), \arctan(4.76), ...",20756,stop
124,qwen_safeguard,24,24,True,A,False,"Okay, let's see. I need to compute the integra...",[E],32463,stop
125,qwen_safeguard,25,25,False,"[18.8105, 11.3449, Yes]",False,"Okay, let's tackle this problem step by step. ...","[18.83, 11.34, True]",18031,stop


In [22]:
debug_df[debug_df["id"] == 1].get('response').values[-1]

'Okay, let\'s try to figure out this integral: the integral from negative infinity to positive infinity of (a^(3/2)) divided by (s² + a²) ds. Hmm, first, I need to recall how to compute integrals of this form. \n\nFirst, let\'s note that a is probably a positive real number since we have a^(3/2) in the numerator. The integral is over all real numbers s. Let me rewrite the integral to make it clearer: a^(3/²) ∫_{-∞}^{+∞} 1/(s² + a²) ds.\n\nI remember that the integral of 1/(x² + b²) from -∞ to ∞ is π/(b). Wait, yes, the standard integral ∫_{-∞}^{∞} 1/(x² + b²) dx = π/b. Let me confirm that. If I let x = b tanθ, then dx = b sec²θ dθ, and the integral becomes ∫_{-π/2}^{π/2} 1/(b² tan²θ + b²) * b sec²θ dθ = (1/b) ∫_{-π/2}^{π/2} sec²θ / (sec²θ) dθ = (1/b) * π. So yes, it\'s π/b.\n\nSo in this case, the denominator is s² + a², so b = a. Therefore, the integral ∫_{-∞}^{∞} 1/(s² + a²) ds = π/a.\n\nWait, but the problem has a^(3/2) multiplied by that integral. So the entire integral is a^(3/2) 

## 8. Summary

Print accuracy broken down by question type.

In [23]:
for variant in PROMPT_VARIANTS:
  print(f"Variant: {variant}")
  mcq_res  = [r for r in results[variant] if r["is_mcq"]]
  free_res = [r for r in results[variant] if not r["is_mcq"]]

  def acc(subset):
      return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

  print("=" * 50)
  print("EVALUATION RESULTS")
  print("=" * 50)
  print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
  print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
  print(f"  Overall    : {sum(r['correct'] for r in results[variant]):4d} / {len(results[variant]):4d}  ({acc(results[variant]):.2f}%)")
  print("=" * 50)

Variant: multiple_answers
EVALUATION RESULTS
  MCQ        :   27 /   38  (71.05%)
  Free-form  :   30 /   62  (48.39%)
  Overall    :   57 /  100  (57.00%)
Variant: qwen_safeguard
EVALUATION RESULTS
  MCQ        :   30 /   38  (78.95%)
  Free-form  :   34 /   62  (54.84%)
  Overall    :   64 /  100  (64.00%)
Variant: frq_blanks
EVALUATION RESULTS
  MCQ        :   29 /   38  (76.32%)
  Free-form  :   30 /   62  (48.39%)
  Overall    :   59 /  100  (59.00%)
Variant: linus
EVALUATION RESULTS
  MCQ        :   28 /   38  (73.68%)
  Free-form  :   34 /   62  (54.84%)
  Overall    :   62 /  100  (62.00%)


## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [24]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

TypeError: string indices must be integers, not 'str'

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!